# Etapa 9. Figuras e tabelas do artigo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/09_figuras_artigo.ipynb)

Reúne, num único lugar, as figuras e tabelas finais do artigo, com o mesmo padrão visual e em 300 dpi. Lê os
resultados gravados no Drive pelas etapas anteriores. Ajuste aqui tamanhos de fonte, cores e rótulos pedidos pela
revista.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

In [ ]:
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mza_oeste as M

plt.rcParams.update({"font.size": 11, "font.family": "serif"})     # padrão de todas as figuras
FIG = SAIDA / "figuras_artigo"
FIG.mkdir(exist_ok=True)

mun = pd.read_csv("dados/municipios_oeste.csv")
malha = gpd.read_file("dados/malha_municipios_oeste.geojson").merge(mun, on="cod_ibge")
gap = pd.read_parquet(SAIDA / "yield_gap.parquet")

## Figura 1. Área de estudo

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
malha.plot(column="dta_mm_m", cmap="Blues", legend=True, edgecolor="grey", lw=0.3, ax=ax,
           legend_kwds={"label": "CAD do solo (mm m$^{-1}$)", "shrink": 0.7})
for _, r in malha[malha["municipio"].isin(["Santa Helena", "Cascavel", "Toledo", "Foz do Iguaçu"])].iterrows():
    ax.annotate(r["municipio"], (r["longitude"], r["latitude"]), fontsize=8, ha="center")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
fig.savefig(FIG / "fig1_area_estudo.png", dpi=300, bbox_inches="tight")
plt.show()

## Figura 5. Mapas das parcelas do yield gap (em % do YG_total)

In [ ]:
parcelas = {"YG_hidrico": "Hídrico inevitável", "YG_data": "Data de semeadura", "YG_manejo": "Manejo"}
media = gap.groupby("cod_ibge")[list(parcelas) + ["YG_total"]].mean()
for p in parcelas:
    media[p + "_%"] = 100 * media[p] / media["YG_total"]
mapa = malha.merge(media, left_on="cod_ibge", right_index=True)

fig, axs = plt.subplots(1, 3, figsize=(16, 5))
for ax, (p, rot) in zip(axs, parcelas.items()):
    mapa.plot(column=p + "_%", cmap="YlOrRd", legend=True, edgecolor="grey", lw=0.3, ax=ax,
              legend_kwds={"label": "% do yield gap total", "shrink": 0.7})
    ax.set_title(rot); ax.set_axis_off()
fig.savefig(FIG / "fig5_mapas_percentual.png", dpi=300, bbox_inches="tight")
plt.show()

## Tabelas

As tabelas são gravadas em CSV para colar no Word ou no Google Docs.

In [ ]:
tab_mun = mun[["municipio", "latitude", "longitude", "altitude_m", "dta_mm_m"]].merge(
    gap.groupby("municipio")["area_colhida_ha"].mean().round(0).rename("area_media_ha"), on="municipio")
tab_mun.round(2).to_csv(FIG / "tabela1_municipios.csv", index=False)

tab_gap = gap.groupby("municipio")[["PP", "PA_cal", "PR", "YG_total"] + list(parcelas)].mean().round(0)
tab_gap.to_csv(FIG / "tabela_gap_municipios.csv")
print(sorted(p.name for p in FIG.iterdir()))

As demais figuras já foram gravadas pelas etapas anteriores, na pasta de resultados:
`fig_serie_produtividade.png` (02), `fig_pp_pa_data.png` (04), `fig_pp_vs_ibge.png` (05),
`fig_superficie_erro.png` e `fig_dispersao_validacao.png` (06), `fig_particao_ano.png` e
`fig_mapas_yield_gap.png` (07), `fig_dssat_vs_mza.png` (08). Se a revista pedir mudanças, altere a célula da
figura no notebook de origem e rode de novo.